---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

---

## Group 24 - AyamNutellaBear

- Geraldo Artemius 13524005
- Mikhael Yonatan 13524051
- Junior Natra Situmorang 13524055
- Reynard Nathanael 13524103
- Nicholas Luis Chandra 13524105

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [52]:
# %pip install -q numpy pandas

## Import Libraries

Every import of the notebook is kept in this cell.

In [53]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## Settings

Paths and shared constants are kept in this class.

In [54]:
class Settings:
    SEED       = 42
    SRC_DIR    = Path('..').resolve()      
    DATA_DIR   = SRC_DIR / 'data'
    OUTPUT_DIR = SRC_DIR / 'output'
    TARGET     = 'is_canceled'
    ID_COL     = 'id'
    VAL_SIZE   = 0.2

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(CFG.SRC_DIR) not in sys.path:
    sys.path.append(str(CFG.SRC_DIR))

TARGET, ID_COL = CFG.TARGET, CFG.ID_COL

## Seed Everything

A fixed seed keeps the results reproducible.

In [55]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [56]:
train_df   = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df    = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})
sample_sub = pd.read_csv(CFG.DATA_DIR / 'sample_submission.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}  Sample submission: {sample_sub.shape}')

Train: (27373, 19)  Test: (6844, 18)  Sample submission: (6844, 2)


---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [57]:
summary = pd.DataFrame({
    'dtype': train_df.dtypes.astype(str),
    'unique': train_df.nunique(),
    'missing': train_df.isna().sum(),
    'min': train_df.min(),
    'max': train_df.max(),
    'value': [train_df[c].dropna().unique()[:5].tolist() for c in train_df.columns],
})
summary

,dtype,unique,missing,min,max,value
id,str,27373,0,0003954f,fff7294f,"[57152f82, af2ec71b, 5d5469ab, d2df0369, 91d25..."
num_adults,float64,5,809,0.0,4.0,"[2.0, 3.0, 1.0, 0.0, 4.0]"
num_children,float64,6,3035,0.0,10.0,"[0.0, 1.0, 2.0, 3.0, 10.0]"
weekend_nights,int64,8,0,0,7,"[1, 2, 0, 4, 3]"
weekday_nights,int64,18,0,0,17,"[2, 1, 5, 3, 0]"
meal_package,str,4,1760,Breakfast,Room Only,"[Breakfast, Half Board, Room Only, Full Board]"
needs_parking,float64,2,1082,0.0,1.0,"[0.0, 1.0]"
room_category,str,7,1002,Type A,Type G,"[Type B, Type D, Type G, Type A, Type C]"
days_before_arrival,float64,407,1123,0.0,448.0,"[88.0, 120.0, 209.0, 93.0, 14.0]"
arrival_year,int64,2,0,2023,2024,"[2024, 2023]"


### Klasifikasi Kolom

In [58]:
NUMERICAL   = ['num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'days_before_arrival', 'prior_cancellations', 'prior_completed_bookings', 'nightly_rate_idr', 'special_requests']
CATEGORICAL = ['meal_package', 'room_category', 'booking_channel']
DATE        = ['arrival_year', 'arrival_month', 'arrival_day']
BINARY      = ['needs_parking', 'is_returning_guest']
FEATURES    = NUMERICAL + CATEGORICAL + DATE + BINARY

assert set(FEATURES) | {ID_COL, TARGET} == set(train_df.columns)
assert set(train_df.columns) - set(test_df.columns) == {TARGET}

**Insights**

- Dataset train berisi 27.373 baris dan test 6.844 baris.
- `id` unik di setiap baris dan hanya digunakan untuk file submission, bukan sebagai fitur.
- Kolom dikelompokkan menjadi numerical, categorical, date, dan binary.

## Explore Data

This part covers distributions and relationships between attributes.

In [59]:
# Distribusi target
print(train_df[TARGET].value_counts())
print(train_df[TARGET].value_counts(normalize=True).round(3))

is_canceled
0    18404
1     8969
Name: count, dtype: int64
is_canceled
0    0.672
1    0.328
Name: proportion, dtype: float64


In [60]:
train_df[NUMERICAL + DATE].describe().T

,count,mean,std,min,25%,50%,75%,max
num_adults,26564.0,1.845242e+00,0.517974,0.0,2.0,2.0,2.0,4.0
num_children,24338.0,1.173884e-01,0.425761,0.0,0.0,0.0,0.0,10.0
weekend_nights,27373.0,8.148541e-01,0.875146,0.0,0.0,1.0,2.0,7.0
weekday_nights,27373.0,2.206517e+00,1.421603,0.0,1.0,2.0,3.0,17.0
days_before_arrival,26250.0,8.537943e+01,85.969638,0.0,17.0,58.0,126.0,448.0
prior_cancellations,27373.0,2.188288e-02,0.356353,0.0,0.0,0.0,0.0,13.0
prior_completed_bookings,27373.0,1.448873e-01,1.693652,0.0,0.0,0.0,0.0,58.0
nightly_rate_idr,25335.0,1.716980e+06,587936.773956,0.0,1338000.0,1644000.0,2019000.0,9165000.0
special_requests,24873.0,6.669481e-01,0.798363,0.0,0.0,0.0,1.0,5.0
arrival_year,27373.0,2.023820e+03,0.384099,2023.0,2024.0,2024.0,2024.0,2024.0


In [61]:
# Distribusi nilai fitur kategorikal dan biner
for c in CATEGORICAL + BINARY:
    print(f'\n{c}:')
    print(train_df[c].value_counts(dropna=False))


meal_package:
meal_package
Breakfast     19654
Room Only      3713
Half Board     2242
NaN            1760
Full Board        4
Name: count, dtype: int64

room_category:
room_category
Type B    20447
Type G     4396
NaN        1002
Type A      699
Type D      514
Type C      189
Type F      119
Type E        7
Name: count, dtype: int64

booking_channel:
booking_channel
Online Travel Agent    17511
Offline Agent           7960
Corporate               1513
Complimentary            301
Airline                   88
Name: count, dtype: int64

needs_parking:
needs_parking
0.0    25497
NaN     1082
1.0      794
Name: count, dtype: int64

is_returning_guest:
is_returning_guest
0    26684
1      689
Name: count, dtype: int64


**Insights**

- Distribusi target 67% tidak cancel dan 33% cancel.
- Fitur numerik cenderung right skewed atau positive skewed. Kolom `prior_cancellations` dan `prior_completed_bookings` kebanyakan bernilai 0.
- Ada kategori yang langka: `Full Board` (4 baris) dan `Type E` (7 baris).

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [62]:
missing = pd.DataFrame({
    'train_missing': train_df.isna().sum(),
    'train_%': (train_df.isna().mean() * 100).round(2),
    'test_missing': test_df.isna().sum(),
    'test_%': (test_df.isna().mean() * 100).round(2),
})
missing[(missing['train_missing'] > 0) | (missing['test_missing'] > 0)]

,train_missing,train_%,test_missing,test_%
days_before_arrival,1123,4.10,290.0,4.24
meal_package,1760,6.43,442.0,6.46
needs_parking,1082,3.95,247.0,3.61
nightly_rate_idr,2038,7.45,540.0,7.89
num_adults,809,2.96,180.0,2.63
num_children,3035,11.09,747.0,10.91
room_category,1002,3.66,242.0,3.54
special_requests,2500,9.13,593.0,8.66


In [63]:
print('Id yang duplikat di train  :', train_df[ID_COL].duplicated().sum())
print('Data yang duplikat di train:', train_df.drop(columns=[ID_COL]).duplicated().sum())

Id yang duplikat di train  : 0
Data yang duplikat di train: 64


In [64]:
check = {
    'Ga ada tamu sama sekali': (train_df['num_adults'] == 0) & (train_df['num_children'] == 0),
    'Ga nginep': (train_df['weekend_nights'] + train_df['weekday_nights']) == 0,
    'Harganya < 0': train_df['nightly_rate_idr'] < 0,
    'Harganya 0 tapi bukan Complimentary': (train_df['nightly_rate_idr'] == 0) & (train_df['booking_channel'] != 'Complimentary'),
    'days_before_arrival < 0': train_df['days_before_arrival'] < 0,
}

for name, mask in check.items():
    print(f'{name:40s}: {mask.sum()}')

dates = pd.to_datetime(dict(year=train_df['arrival_year'], month=train_df['arrival_month'], day=train_df['arrival_day']), errors='coerce')
print('Data tanggal tidak valid:', dates.isna().sum())

Ga ada tamu sama sekali                 : 0
Ga nginep                               : 64
Harganya < 0                            : 0
Harganya 0 tapi bukan Complimentary     : 146
days_before_arrival < 0                 : 0
Data tanggal tidak valid: 0


In [65]:
# Deteksi outlier 
for c in NUMERICAL:
    q1, q3 = train_df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas  = q3 + 1.5 * iqr
    n_outlier = ((train_df[c] < batas_bawah) | (train_df[c] > batas_atas)).sum()
    print(f'{c:26s} IQR={iqr:>10.1f}  batas=[{batas_bawah:.1f}, {batas_atas:.1f}]  outlier={n_outlier}')

num_adults                 IQR=       0.0  batas=[2.0, 2.0]  outlier=7400
num_children               IQR=       0.0  batas=[0.0, 0.0]  outlier=2020
weekend_nights             IQR=       2.0  batas=[-3.0, 5.0]  outlier=16
weekday_nights             IQR=       2.0  batas=[-2.0, 6.0]  outlier=258
days_before_arrival        IQR=     109.0  batas=[-146.5, 289.5]  outlier=916
prior_cancellations        IQR=       0.0  batas=[0.0, 0.0]  outlier=241
prior_completed_bookings   IQR=       0.0  batas=[0.0, 0.0]  outlier=600
nightly_rate_idr           IQR=  681000.0  batas=[316500.0, 3040500.0]  outlier=1144
special_requests           IQR=       1.0  batas=[-1.5, 2.5]  outlier=563


In [66]:
print(train_df['num_children'].value_counts().sort_index())
print(train_df['weekday_nights'].value_counts().sort_index().tail())

num_children
0.0     22318
1.0      1221
2.0       780
3.0        16
9.0         2
10.0        1
Name: count, dtype: int64
weekday_nights
13    3
14    6
15    9
16    2
17    2
Name: count, dtype: int64


**Insights**

- Missing values ada di 8 kolom (sekitar 3–11%) dengan proporsi yang mirip di train dan test.
- Duplikasi tidak ada `id` ganda, tetapi ada 64 baris yang identik di semua kolom lain.
- Harga 0 tetapi bukan Complimentary: 146 baris.
- Tidak ada tamu 0, harga negatif, `days_before_arrival` negatif, ataupun tanggal yang tidak valid.
- Kolom dengan IQR = 0 (`num_adults`, `num_children`, `prior_cancellations`, `prior_completed_bookings`) tidak dapat dinilai dengan IQR, karena semua nilai selain nilai paling umum akan dianggap outlier. 
- Ada nilai yang tidak realistis yaitu `num_children` = 9 dan 10, yang melompat jauh dari nilai 3.

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [67]:
train_set, val_set = train_test_split(
    train_df,
    test_size=CFG.VAL_SIZE,
    stratify=train_df[TARGET],
    random_state=CFG.SEED,
)

train_set = train_set.reset_index(drop=True)
val_set   = val_set.reset_index(drop=True)
test_set  = test_df.copy()

print(f'Train: {train_set.shape}  Val: {val_set.shape}  Test: {test_set.shape}')
print('Proporsi target train:', train_set[TARGET].value_counts(normalize=True).round(3).to_dict())
print('Proporsi target val  :', val_set[TARGET].value_counts(normalize=True).round(3).to_dict())

Train: (21898, 19)  Val: (5475, 19)  Test: (6844, 18)
Proporsi target train: {0: 0.672, 1: 0.328}
Proporsi target val  : {0: 0.672, 1: 0.328}


**Insights**

- Split dilakukan secara *stratified* (stratified random sampling) sehingga proporsi kelas 67:33 sama di train dan validation.
- Semua statistik pada tahap Data Preparation dihitung dari `train_set` saja.

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [68]:
print('Jumlah fitur yang dipakai:', len(FEATURES))
print('Fitur:', FEATURES)
print('Bukan fitur:', [ID_COL, TARGET])

Jumlah fitur yang dipakai: 17
Fitur: ['num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'days_before_arrival', 'prior_cancellations', 'prior_completed_bookings', 'nightly_rate_idr', 'special_requests', 'meal_package', 'room_category', 'booking_channel', 'arrival_year', 'arrival_month', 'arrival_day', 'needs_parking', 'is_returning_guest']
Bukan fitur: ['id', 'is_canceled']


**Insights**

- Semua 17 fitur digunakan. 
- `id` tidak dipakai sebagai fitur karena hanya pengenal unik, tetapi tetap disimpan untuk file submission.
- Tidak dilakukan sampling karena ukuran data masih kecil untuk diproses. Seleksi record hanya berupa penghapusan duplikat di `train_set`.

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Duplicates

This part covers repeated records.

In [69]:
before = len(train_set)

train_set = train_set.drop_duplicates(subset=train_set.columns.drop(ID_COL), keep='first').reset_index(drop=True)

print(f'Duplikat dihapus dari train_set: {before - len(train_set)}')
print(f'Jumlah baris train_set sekarang: {train_set.shape}')

Duplikat dihapus dari train_set: 40
Jumlah baris train_set sekarang: (21858, 19)


**Insights**

- Duplikat dihapus karena baris yang identik di semua kolom (selain `id`) membuat pola tertentu tercatat lebih dari sekali. Baris yang duplikat di drop agar tidak menimbulkan bias pada hasil analisis dan tidak membuat overfitting.
- Penghapusan hanya dilakukan di `train_set`. 

### Missing Values

This part covers missing values, if any.

In [ ]:
NUMERICAL_MISSING   = ['num_adults', 'num_children', 'days_before_arrival', 'nightly_rate_idr', 'special_requests']
CATEGORICAL_MISSING = ['meal_package', 'room_category', 'needs_parking']

fill = {c: train_set[c].median() for c in NUMERICAL_MISSING}
fill.update({c: train_set[c].mode().iloc[0] for c in CATEGORICAL_MISSING})
print(fill)

train_set = train_set.fillna(fill)
val_set   = val_set.fillna(fill)
test_set  = test_set.fillna(fill)

for name, df in [('train', train_set), ('val', val_set), ('test', test_set)]:
    print(f'Sisa missing {name}: {df.isna().sum().sum()}')

{'num_adults': np.float64(2.0), 'num_children': np.float64(0.0), 'days_before_arrival': np.float64(57.0), 'nightly_rate_idr': np.float64(1647000.0), 'special_requests': np.float64(0.0), 'meal_package': 'Breakfast', 'room_category': 'Type B', 'needs_parking': np.float64(0.0)}
Sisa missing train: 0
Sisa missing val: 0
Sisa missing test: 0


**Insights**

- Data Numerik diisi median. Jika pakai mean, akan ikut tertarik oleh nilai ekstrem, sedangkan median tidak.
- Data Kategorikal dan biner diisi modus. Karena string tidak memiliki rata-rata atau nilai tengah. 

### Outliers

This part covers values far from the rest of the data.

In [ ]:
child_median = train_set['num_children'].median()   

for name, df in [('train', train_set), ('val', val_set), ('test', test_set)]:
    print(f'{name} num_children > 3: {(df["num_children"] > 3).sum()}')
    df.loc[df['num_children'] > 3, 'num_children'] = child_median

train num_children > 3: 3
val num_children > 3: 0
test num_children > 3: 0


**Insights**

- Outlier (`num_children` ≥ 9) diganti dengan nilai median dari `train_set`. Menggunakan batas > 3 dipilih karena nilai 3.

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
INT_COLS = ['num_adults', 'num_children', 'special_requests', 'days_before_arrival', 'needs_parking']

for df in (train_set, val_set, test_set):
    df[INT_COLS] = df[INT_COLS].astype(int)

In [92]:
for name, df in [('train', train_set), ('val', val_set), ('test', test_set)]:
    print(f'{name:5s} shape={df.shape}  missing={df.isna().sum().sum()}  num_children max={df["num_children"].max()}')

print('Baris identik setelah imputasi di train_set:', train_set.drop(columns=[ID_COL]).duplicated().sum())
train_set.dtypes

train shape=(21858, 19)  missing=0  num_children max=3
val   shape=(5475, 19)  missing=0  num_children max=3
test  shape=(6844, 18)  missing=0  num_children max=3
Baris identik setelah imputasi di train_set: 54


id                              str
num_adults                    int64
num_children                  int64
weekend_nights                int64
weekday_nights                int64
meal_package                    str
needs_parking                 int64
room_category                   str
days_before_arrival           int64
arrival_year                  int64
arrival_month                 int64
arrival_day                   int64
booking_channel                 str
is_returning_guest            int64
prior_cancellations           int64
prior_completed_bookings      int64
nightly_rate_idr            float64
special_requests              int64
is_canceled                   int64
dtype: object

**Insights**

- Baris dengan 0 malam dan baris dengan harga 0 di luar channel Complimentary tidak diubah.
- Baris yang menjadi identik setelah imputasi tidak dihapus, karena kemiripannya berasal dari nilai hasil imputasi, bukan dari data asli.

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [74]:
# Write your code here

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [75]:
# Write your code here

### Encoding

This part covers the numeric representation of categorical attributes.

In [76]:
# Write your code here

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [77]:
# Write your code here

## Class Balance

This part covers the proportion of each class in the training data.

In [78]:
# Write your code here

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [79]:
# Write your code here

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [80]:
# Write your code here

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [81]:
# Write your code here

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [82]:
# Write your code here

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [83]:
# Write your code here

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [84]:
# Write your code here

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [85]:
# Write your code here

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [86]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [87]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [88]:
# Write your code here

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [89]:
# Write your code here

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [90]:
# Write your code here

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.